# **Twilio WhatsApp Sandbox — Send messages**

In [ ]:
!pip install twilio --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 21.7 MB/s eta 0:00:00


In [ ]:
#  Example 5A — Send a simple WhatsApp message via Twilio Sandbox

import os
from getpass import getpass
from twilio.rest import Client

# --- Twilio credentials ---
TW_SID = os.environ.get('TWILIO_SID')
TW_TOKEN = os.environ.get('TWILIO_TOKEN')

if not TW_SID or not TW_TOKEN:
    print("Enter your Twilio Account SID and Auth Token:")
    TW_SID = getpass("Twilio SID: ")
    TW_TOKEN = getpass("Twilio Token: ")
    os.environ['TWILIO_SID'] = TW_SID
    os.environ['TWILIO_TOKEN'] = TW_TOKEN

# --- Twilio WhatsApp Sandbox sender ---
FROM = 'whatsapp:+14155238886'  # Standard Twilio Sandbox number

# --- User's WhatsApp number ---
TO = input("Enter your WhatsApp number (e.g. +91XXXXXXXXXX): ").strip()

#  Add whatsapp: prefix if missing
if not TO.startswith('whatsapp:'):
    TO = f'whatsapp:{TO}'

# --- Initialize Twilio client ---
client = Client(TW_SID, TW_TOKEN)

# --- Send message ---
msg = client.messages.create(
    body='Hello from Twilio WhatsApp Sandbox — test message!',
    from_=FROM,
    to=TO
)
print(' Message sent successfully!')
print('Message SID:', msg.sid)

Enter your Twilio Account SID and Auth Token:
Twilio SID: ··········
Twilio Token: ··········
Enter your WhatsApp number (e.g. +91XXXXXXXXXX): 918754278349
 Message sent successfully!
Message SID: SM18e7d6a298a264b53dcdfcdaa17406fa


# **Send weather update via Twilio WhatsApp Sandbox + OpenWeatherMap**

In [ ]:
#  Example 5B — Send weather update via Twilio WhatsApp Sandbox + OpenWeatherMap
!pip install twilio --quiet

import os, requests
from getpass import getpass
from twilio.rest import Client

# --- Environment variables ---
TW_SID = os.environ.get('TWILIO_SID')
TW_TOKEN = os.environ.get('TWILIO_TOKEN')
OWM_KEY = os.environ.get('OWM_API_KEY')

if not (TW_SID and TW_TOKEN and OWM_KEY):
    print(" Missing TWILIO or OWM keys. Please enter them below:")
    if not TW_SID:
        TW_SID = getpass("Enter Twilio SID: ")
        os.environ['TWILIO_SID'] = TW_SID
    if not TW_TOKEN:
        TW_TOKEN = getpass("Enter Twilio Auth Token: ")
        os.environ['TWILIO_TOKEN'] = TW_TOKEN
    if not OWM_KEY:
        OWM_KEY = getpass("Enter OpenWeatherMap API Key: ")
        os.environ['OWM_API_KEY'] = OWM_KEY

# --- Twilio WhatsApp Sandbox number ---
FROM = 'whatsapp:+14155238886'

# --- User inputs ---
TO = input("Enter your WhatsApp number (e.g. +91XXXXXXXXXX): ").strip()
city = input("Enter city for weather update (e.g. Mumbai): ").strip()

#  Ensure 'whatsapp:' prefix
if not TO.startswith('whatsapp:'):
    TO = f'whatsapp:{TO}'

# --- Fetch weather ---
url = f"http://api.openweathermap.org/data/2.5/weather?q={city}&appid={OWM_KEY}&units=metric"
r = requests.get(url)

if not r.ok:
    print(f" Weather fetch failed: {r.status_code} - {r.text}")
else:
    d = r.json()
    temp = d.get("main", {}).get("temp")
    desc = d.get("weather", [{}])[0].get("description", "").capitalize()
    body = f" Weather update for {d.get('name')}: {temp}°C, {desc}"

    # --- Send message ---
    client = Client(TW_SID, TW_TOKEN)
    msg = client.messages.create(body=body, from_=FROM, to=TO)

    print(" Message sent successfully!")
    print("Message SID:", msg.sid)

Enter your WhatsApp number (e.g. +91XXXXXXXXXX): 918754278349
Enter city for weather update (e.g. Mumbai): mumbai
 Weather fetch failed: 401 - {"cod":401, "message": "Invalid API key. Please see https://openweathermap.org/faq#error401 for more info."}


# **Leetcode Solver Agent --> Whatsapp ( AI Agent + Twilio Agent)**

In [ ]:
!pip install langchain langchain-groq --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.8 MB/s eta 0:00:00


In [ ]:
import os
from getpass import getpass
from twilio.rest import Client
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate

# ==============================
#  Load Groq API Key
# ==============================
GROQ_KEY = os.environ.get("GROQ_API_KEY")
if not GROQ_KEY:
    GROQ_KEY = getpass("Enter GROQ API Key: ").strip()
    os.environ["GROQ_API_KEY"] = GROQ_KEY

# ==============================
#  Load Twilio Credentials
# ==============================
TW_SID = os.environ.get("TWILIO_SID")
TW_TOKEN = os.environ.get("TWILIO_TOKEN")

if not (TW_SID and TW_TOKEN):
    TW_SID = getpass("Twilio SID: ").strip()
    TW_TOKEN = getpass("Twilio Auth Token: ").strip()
    os.environ["TWILIO_SID"] = TW_SID
    os.environ["TWILIO_TOKEN"] = TW_TOKEN

# Twilio WhatsApp Sandbox
FROM = "whatsapp:+14155238886"

# ==============================
#  User Inputs
# ==============================
TO = input("Enter your WhatsApp number (+91XXXXXXXXXX): ").strip()
if not TO.startswith("whatsapp:"):
    TO = f"whatsapp:{TO}"

problem = input("Enter LeetCode problem description or title: ").strip()

# ==============================
#  Groq LLM (LeetCode Solver)
# ==============================
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.2,
)

prompt = PromptTemplate.from_template("""
You are an expert LeetCode problem solver.

Given a problem:
{problem}

Return:
1. Short explanation
2. Optimized Python solution
3. Example outputs

Keep it concise and WhatsApp-friendly.
""")

chain = prompt | llm

result = chain.invoke({"problem": problem})
solution_text = result.content

# ==============================
#  Send Solution via WhatsApp
# ==============================
try:
    client = Client(TW_SID, TW_TOKEN)
    message = client.messages.create(
        body=f" *LeetCode Solver*\n\n{solution_text}",
        from_=FROM,
        to=TO
    )
    print(" WhatsApp message sent!")
    print("Message SID:", message.sid)

except Exception as e:
    print(" Error sending WhatsApp message:", e)

Enter GROQ API Key: ··········
Enter your WhatsApp number (+91XXXXXXXXXX): 918754278349
Enter LeetCode problem description or title: two sum
 WhatsApp message sent!
Message SID: SM1cd05ebdea70e25b3c75308ad66a5ed3
